# CUDA-Only Unsloth Banking77 Generalization Training

This notebook fine-tunes an Unsloth LoRA model on [`tsilva/banking77`](https://huggingface.co/datasets/tsilva/banking77) for intent classification. The goal is no longer memorization; the goal is **test-set generalization**.

The data contract is:

- `system_prompt` -> system message
- `text` -> user prompt
- `label_text` -> assistant response

The training flow is:

- stratified train/validation split from the Hugging Face `train` split
- supervised fine-tuning with `SFTTrainer`
- early stopping on validation loss
- best-checkpoint restore
- deterministic generation on the Hugging Face `test` split
- exact-match test accuracy and W&B prediction logging

It is intentionally **CUDA-only**. If CUDA, Unsloth, TRL, W&B, or the model stack is missing, the notebook stops instead of falling back to CPU, MPS, or a toy local model.


## 1. Environment Contract

Run this in a Linux or Windows CUDA environment with Unsloth installed. The repo's Modal runner is the intended validation path for this notebook.

```bash
./run_modal.sh notebooks/unsloth-minimal-training.ipynb
```

W&B logging expects the Modal secret `wandb-secret` to provide `WANDB_API_KEY`.


### Configuration

The defaults favor accuracy over speed while staying within the Modal A10 runtime budget. `test_eval_limit=None` means the final accuracy is computed on the full Banking77 test split.


In [ ]:
CONFIG = {
    # Reproducibility
    'seed': 3407,

    # Data
    'dataset_name': 'tsilva/banking77',
    'train_split': 'train',
    'test_split': 'test',
    'val_fraction': 0.1,
    'train_limit': None,  # Set to an int for quick debugging; None uses the full train split
    'test_eval_limit': None,  # None evaluates all 3,076 test rows

    # Model
    'model_name': 'unsloth/gemma-3-1b-it-unsloth-bnb-4bit',
    'max_seq_length': 2048,
    'load_in_4bit': True,
    'load_in_8bit': False,
    'full_finetuning': False,

    # LoRA
    'lora_rank': 32,
    'lora_alpha': 32,
    'lora_dropout': 0,

    # Training
    'per_device_train_batch_size': 2,
    'gradient_accumulation_steps': 4,
    'max_steps': 1600,
    'learning_rate': 2e-4,
    'warmup_ratio': 0.03,
    'logging_steps': 25,
    'eval_steps': 200,
    'early_stopping_patience': 3,
    'output_dir': 'outputs/unsloth-banking77-generalization',

    # Experiment tracking
    'wandb_project': 'unsloth-minimal-training',
    'wandb_run_name': 'unsloth-banking77-generalization',

    # Inference
    'max_new_tokens': 16,
}


### Hard CUDA and Package Checks

This cell is deliberately strict. A CUDA-only notebook should fail early and plainly on unsupported machines.


In [ ]:
import importlib.util
import platform
import sys

import torch

missing = [
    package
    for package in ['unsloth', 'trl', 'datasets', 'transformers', 'bitsandbytes', 'wandb']
    if importlib.util.find_spec(package) is None
]
if missing:
    raise RuntimeError(
        'Missing CUDA fine-tuning packages: '
        + ', '.join(missing)
        + '. Install them in a CUDA environment before running this notebook.'
    )

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required. This notebook intentionally has no CPU or MPS fallback.')

if torch.cuda.get_device_capability(0)[0] < 7:
    raise RuntimeError('Unsloth Core requires CUDA capability 7.0 or newer.')

print(f'Python: {sys.version.split()[0]}')
print(f'Platform: {platform.platform()}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.version.cuda}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'CUDA capability: {torch.cuda.get_device_capability(0)}')


### Import Unsloth Primitives

Importing Unsloth before other model libraries lets it patch kernels and model classes as intended.


In [ ]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from trl import SFTConfig, SFTTrainer


## 2. Load Banking77 and Build Splits

The Hugging Face `train` split is split again into train and validation partitions using a deterministic stratified split over the integer label column. The original Hugging Face `test` split is kept untouched for final accuracy.


In [ ]:
import random
from collections import defaultdict

from datasets import Dataset, load_dataset

raw_dataset = load_dataset(CONFIG['dataset_name'])
source_train = raw_dataset[CONFIG['train_split']]
test_dataset = raw_dataset[CONFIG['test_split']]

label_to_indices = defaultdict(list)
for index, label in enumerate(source_train['label']):
    label_to_indices[int(label)].append(index)

rng = random.Random(CONFIG['seed'])
train_indices = []
val_indices = []
for label, indices in sorted(label_to_indices.items()):
    shuffled = list(indices)
    rng.shuffle(shuffled)
    val_count = max(1, round(len(shuffled) * CONFIG['val_fraction']))
    val_indices.extend(shuffled[:val_count])
    train_indices.extend(shuffled[val_count:])

rng.shuffle(train_indices)
rng.shuffle(val_indices)

if CONFIG['train_limit'] is not None:
    train_indices = train_indices[:CONFIG['train_limit']]

train_dataset = source_train.select(train_indices)
val_dataset = source_train.select(val_indices)
if CONFIG['test_eval_limit'] is not None:
    test_dataset = test_dataset.select(range(CONFIG['test_eval_limit']))

LABELS = sorted(set(source_train['label_text']))

print(raw_dataset)
print(f'Train rows: {len(train_dataset):,}')
print(f'Validation rows: {len(val_dataset):,}')
print(f'Test rows: {len(test_dataset):,}')
print(f'Labels: {len(LABELS)}')
print(train_dataset[0])


## 3. Load a Real Unsloth Model

The default model is Gemma 3 1B Instruct in Unsloth 4-bit form. That is still compact enough for the Modal A10, but more capable than the 270M smoke-test model.


In [ ]:
model, tokenizer = FastModel.from_pretrained(
    model_name=CONFIG['model_name'],
    max_seq_length=CONFIG['max_seq_length'],
    load_in_4bit=CONFIG['load_in_4bit'],
    load_in_8bit=CONFIG['load_in_8bit'],
    full_finetuning=CONFIG['full_finetuning'],
)

tokenizer = get_chat_template(
    tokenizer,
    chat_template='gemma3',
)


### Attach LoRA Adapters

This is the Unsloth adapter primitive. The base model stays frozen while selected projection modules receive trainable low-rank matrices.


In [ ]:
model = FastModel.get_peft_model(
    model,
    r=CONFIG['lora_rank'],
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=CONFIG['seed'],
    use_rslora=False,
    loftq_config=None,
)


### Inspect Trainable Parameters

This confirms we are training a LoRA adapter instead of full fine-tuning the base model.


In [ ]:
trainable_params = 0
total_params = 0
for param in model.parameters():
    total_params += param.numel()
    if param.requires_grad:
        trainable_params += param.numel()

print(f'Trainable parameters: {trainable_params:,}')
print(f'Total parameters: {total_params:,}')
print(f'Trainable fraction: {100 * trainable_params / total_params:.3f}%')
assert 0 < trainable_params < total_params


## 4. Format the Dataset with the Chat Template

Every row becomes a three-message conversation: system instructions, user text, and assistant intent label.


In [ ]:
def to_messages(row):
    return {
        'messages': [
            {'role': 'system', 'content': row['system_prompt']},
            {'role': 'user', 'content': row['text']},
            {'role': 'assistant', 'content': row['label_text']},
        ]
    }

def apply_template(row):
    return {
        'text': tokenizer.apply_chat_template(
            row['messages'],
            tokenize=False,
            add_generation_prompt=False,
        ).removeprefix('<bos>')
    }

formatted_train_dataset = train_dataset.map(to_messages).map(apply_template)
formatted_val_dataset = val_dataset.map(to_messages).map(apply_template)
print(formatted_train_dataset[0]['text'][:1200])


## 5. Build the SFT Trainer with Early Stopping

Validation loss is evaluated every `eval_steps`. Early stopping watches that validation loss and `load_best_model_at_end=True` restores the best checkpoint before final test evaluation.


In [ ]:
import os

import wandb
from transformers import EarlyStoppingCallback, TrainerCallback

os.environ['WANDB_PROJECT'] = CONFIG['wandb_project']
os.environ.setdefault('WANDB_LOG_MODEL', 'false')

wandb_run = wandb.init(
    project=CONFIG['wandb_project'],
    name=CONFIG['wandb_run_name'],
    config=CONFIG,
)
wandb_run_id = wandb_run.id

class WandbScalarLogger(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs or wandb.run is None:
            return
        numeric_logs = {
            key: value
            for key, value in logs.items()
            if isinstance(value, (int, float))
        }
        if numeric_logs:
            wandb.log(numeric_logs, step=state.global_step)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_train_dataset,
    eval_dataset=formatted_val_dataset,
    args=SFTConfig(
        dataset_text_field='text',
        per_device_train_batch_size=CONFIG['per_device_train_batch_size'],
        gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
        max_steps=CONFIG['max_steps'],
        learning_rate=CONFIG['learning_rate'],
        warmup_ratio=CONFIG['warmup_ratio'],
        logging_steps=CONFIG['logging_steps'],
        eval_strategy='steps',
        eval_steps=CONFIG['eval_steps'],
        save_strategy='steps',
        save_steps=CONFIG['eval_steps'],
        save_total_limit=2,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        load_best_model_at_end=True,
        optim='adamw_8bit',
        weight_decay=0.001,
        lr_scheduler_type='linear',
        seed=CONFIG['seed'],
        output_dir=CONFIG['output_dir'],
        report_to='none',
        run_name=CONFIG['wandb_run_name'],
    ),
    callbacks=[
        WandbScalarLogger(),
        EarlyStoppingCallback(early_stopping_patience=CONFIG['early_stopping_patience']),
    ],
)


### Mask Prompt Tokens

This Unsloth helper trains only on assistant labels. The system and user messages still condition the model, but they do not contribute to the loss.


In [ ]:
trainer = train_on_responses_only(
    trainer,
    instruction_part='<start_of_turn>user\n',
    response_part='<start_of_turn>model\n',
)


### Show CUDA Memory Before Training

This gives us a concrete read on the GPU budget before the training loop starts.


In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory_gb = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
max_memory_gb = round(gpu_stats.total_memory / 1024**3, 3)
print(f'GPU: {gpu_stats.name}')
print(f'Max memory: {max_memory_gb} GB')
print(f'Reserved before training: {start_gpu_memory_gb} GB')


## 6. Train

The trainer evaluates validation loss periodically, early-stops when it stops improving, and restores the best checkpoint at the end.


In [ ]:
train_result = trainer.train()

if wandb.run is not None:
    wandb.log(
        {
            f'train_result/{key}': value
            for key, value in train_result.metrics.items()
            if isinstance(value, (int, float))
        },
        step=train_result.global_step,
    )

print(f'Best checkpoint: {trainer.state.best_model_checkpoint}')
print(f'Best eval loss: {trainer.state.best_metric}')
train_result


### Show CUDA Memory After Training

A small memory summary helps verify that the run stayed within the target GPU budget.


In [ ]:
end_gpu_memory_gb = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
used_for_training_gb = round(end_gpu_memory_gb - start_gpu_memory_gb, 3)
print(f'Reserved after training: {end_gpu_memory_gb} GB')
print(f'Additional reserved during training: {used_for_training_gb} GB')


## 7. Final Test Accuracy

The test set is evaluated once after training. Generation is deterministic and normalized to the allowed label set before exact-match scoring.


In [ ]:
def normalize_label(text: str) -> str:
    cleaned = text.strip().strip('`').strip().strip('.,;:!')
    first_token = cleaned.split()[0] if cleaned.split() else ''
    first_token = first_token.strip('`').strip().strip('.,;:!')
    if cleaned in LABELS:
        return cleaned
    if first_token in LABELS:
        return first_token
    return first_token

@torch.inference_mode()
def predict_label(row) -> str:
    messages = [
        {'role': 'system', 'content': row['system_prompt']},
        {'role': 'user', 'content': row['text']},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    ).removeprefix('<bos>')
    inputs = tokenizer(text, return_tensors='pt').to('cuda')
    prompt_length = inputs['input_ids'].shape[-1]
    outputs = model.generate(
        **inputs,
        max_new_tokens=CONFIG['max_new_tokens'],
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated_ids = outputs[0, prompt_length:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

prediction_rows = []
correct = 0
for index, row in enumerate(test_dataset):
    prediction = predict_label(row)
    normalized_prediction = normalize_label(prediction)
    exact_match = normalized_prediction == row['label_text']
    correct += int(exact_match)
    prediction_rows.append(
        {
            'index': index,
            'text': row['text'],
            'expected_response': row['label_text'],
            'prediction': prediction,
            'normalized_prediction': normalized_prediction,
            'exact_match': exact_match,
        }
    )
    if (index + 1) % 250 == 0:
        print(f'Evaluated {index + 1:,}/{len(test_dataset):,} test rows')

test_accuracy = correct / len(test_dataset)
print(f'Test accuracy: {test_accuracy:.2%} ({correct:,}/{len(test_dataset):,})')

for row in prediction_rows[:20]:
    print('-' * 80)
    print(f"Text:        {row['text']}")
    print(f"Expected:    {row['expected_response']}")
    print(f"Prediction:  {row['prediction']}")
    print(f"Normalized:  {row['normalized_prediction']}")
    print(f"Match:       {row['exact_match']}")


### Log Test Predictions to W&B

The prediction table lets us inspect misses directly in W&B, while scalar metrics make runs comparable.


In [ ]:
if wandb.run is None:
    wandb.init(
        project=CONFIG['wandb_project'],
        id=wandb_run_id,
        resume='allow',
    )

predictions_table = wandb.Table(
    columns=[
        'index',
        'text',
        'expected_response',
        'prediction',
        'normalized_prediction',
        'exact_match',
    ]
)
for row in prediction_rows:
    predictions_table.add_data(
        row['index'],
        row['text'],
        row['expected_response'],
        row['prediction'],
        row['normalized_prediction'],
        row['exact_match'],
    )

wandb.log({
    'test/predictions': predictions_table,
    'test/accuracy': test_accuracy,
    'test/correct': correct,
    'test/total': len(test_dataset),
    'best/eval_loss': trainer.state.best_metric,
})
print(
    f"Logged {len(prediction_rows)} test predictions to W&B project "
    f"{CONFIG['wandb_project']} run {wandb_run_id}"
)


## 8. Save the LoRA Adapter

Saving only the adapter keeps the artifact small. Load it later on top of the same base model for inference or continued fine-tuning.


In [ ]:
adapter_dir = 'outputs/unsloth-banking77-generalization-lora'
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f'Saved LoRA adapter to {adapter_dir}')

import wandb

if wandb.run is not None:
    wandb.finish()


## Key Takeaways

- This notebook uses real Unsloth primitives and no fallback training path.
- The dataset source is `tsilva/banking77`: `system_prompt` -> system, `text` -> user, `label_text` -> assistant.
- The Hugging Face train split is split into train and validation partitions.
- Early stopping monitors validation loss and the best checkpoint is restored before test evaluation.
- Final accuracy is computed on the Hugging Face test split with deterministic generation.
- W&B logs scalar training/eval metrics and a full test prediction table.
